# Célula 1 – Configuração Geral (Imports e Constantes)



In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms, models
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from tqdm.notebook import tqdm
from collections import Counter
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# --- Constantes e Configurações Globais ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BASE_DIR = "../../data/final" # ATENÇÃO: Verifique se este caminho está correto
REGIONS = ["forehead", "chin", "nose", "left_cheek", "right_cheek"]
region_to_idx = {r: i for i, r in enumerate(REGIONS)}
CLASS_NAMES = ['Grau 1', 'Grau 2', 'Grau 3', 'Grau 4']
BATCH_SIZE = 16
NUM_EPOCHS = 40

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(25),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.ToTensor(),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

print(f"Ambiente configurado para usar o dispositivo: {device}")

# Célula 2 – Definição da Classe de Dataset


In [ ]:

class AcneDataset(Dataset):
    """Classe de Dataset genérica para carregar imagens de treino, val ou teste."""
    def __init__(self, base_dir, splits, transform=None, include_val_in_train=False):
        self.samples = []
        self.transform = transform
        
        if include_val_in_train and 'train' in splits:
            splits.append('val')

        for region in REGIONS:
            for split in splits:
                split_dir = os.path.join(base_dir, region, split)
                if os.path.isdir(split_dir):
                    ds = datasets.ImageFolder(split_dir, transform=None)
                    for path, label in ds.samples:
                        self.samples.append((path, label, region))
    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label, region = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        region_idx = region_to_idx[region]
        return img, label, region_idx

print("Classe AcneDataset definida.")

# Célula 3 – Arquitetura do Modelo 


In [ ]:

class OptimizedMultiRegionResNet18(nn.Module):
    def __init__(self, num_classes=4, num_regions=5):
        super().__init__()
        self.resnet = models.resnet18(weights=None)
        in_features = self.resnet.fc.in_features
        self.resnet.fc = nn.Identity()
        
        self.region_embed = nn.Embedding(num_regions, 64)
        
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(in_features + 64, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, num_classes)
        )

    def forward(self, x, region_idx):
        img_features = self.resnet(x)
        region_features = self.region_embed(region_idx.long()).squeeze(1)
        combined = torch.cat([img_features, region_features], dim=1)
        return self.classifier(combined)

print("Definição da classe do modelo corrigida e pronta.")

# Célula 4 – Carregamento dos Dados


In [ ]:
train_ds = AcneDataset(BASE_DIR, splits=['train'], transform=train_transform, include_val_in_train=True)
test_ds = AcneDataset(BASE_DIR, splits=['test'], transform=val_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

print(f"DataLoaders de treino ({len(train_ds)} imagens) e teste ({len(test_ds)} imagens) criados.")

# Célula 5 – Funções de Treinamento


In [ ]:

def train_model(model, train_loader, test_loader, criterion, optimizer, scheduler, num_epochs):
    best_acc = 0.0
    
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        for imgs, labels, regions in train_loader:
            imgs, labels, regions = imgs.to(device), labels.to(device), regions.to(device)
            optimizer.zero_grad()
            outputs = model(imgs, regions)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        # Validação
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for imgs, labels, regions in test_loader:
                imgs, labels, regions = imgs.to(device), labels.to(device), regions.to(device)
                outputs = model(imgs, regions)
                _, predicted = outputs.max(1)
                total += labels.size(0)
                correct += predicted.eq(labels).sum().item()
        
        epoch_acc = 100. * correct / total
        epoch_loss = running_loss / len(train_loader)
        
        print(f"Epoch {epoch+1}/{num_epochs} | Loss: {epoch_loss:.4f} | Test Acc: {epoch_acc:.2f}%")
        
        scheduler.step(epoch_acc)
        
        if epoch_acc > best_acc:
            best_acc = epoch_acc
            torch.save(model.state_dict(), "best_robust_model.pth")
            print(f"  -> Modelo salvo com acurácia de {best_acc:.2f}%")

    print("\n=== TREINAMENTO FINALIZADO ===")

# Célula 6 – Execução do Treinamento


In [ ]:
model = OptimizedMultiRegionResNet18().to(device)

all_labels = [label for _, label, _ in train_ds]
class_weights = compute_class_weight(class_weight="balanced", classes=np.unique(all_labels), y=all_labels)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5, verbose=True)

train_model(model, train_loader, test_loader, criterion, optimizer, scheduler, num_epochs=NUM_EPOCHS)

# Célula 7 – Relatório de Performance Principal

In [ ]:

print("--- Iniciando Análise de Performance Principal ---")
model_to_test = OptimizedMultiRegionResNet18().to(device)
model_to_test.load_state_dict(torch.load("best_robust_model.pth"))
model_to_test.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels, regions in test_loader:
        imgs, labels, regions = imgs.to(device), labels.to(device), regions.to(device)
        outputs = model_to_test(imgs, regions)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print("\n" + "="*50)
print("RELATÓRIO DE CLASSIFICAÇÃO DETALHADO")
print("="*50)
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))

# Célula 8 – Tabela de Estabilidade das Métricas (Média ± Desvio Padrão)

In [ ]:

print("Iniciando análise de estabilidade com ambiente controlado...")

import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix
from tqdm.notebook import tqdm
from PIL import Image

CLASS_NAMES = ['Grau 1', 'Grau 2', 'Grau 3', 'Grau 4'] 
NUM_RUNS = 30 

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

print("Criando um DataLoader de teste seguro com as transformações corretas...")
test_ds_safe = AcneDataset(BASE_DIR, splits=['test'], transform=val_transform)
test_loader_safe = DataLoader(test_ds_safe, batch_size=16, shuffle=False)
print("DataLoader seguro criado.")

model_to_test = OptimizedMultiRegionResNet18()
model_to_test.load_state_dict(torch.load("best_robust_model.pth"))
model_to_test.to(device)
model_to_test.eval()

eval_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomRotation(15),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor()
])

def run_full_metric_analysis(model, loader, num_runs):
    results_history = {name: {'precision': [], 'recall': [], 'f1-score': [], 'accuracy': []} for name in CLASS_NAMES}
    accuracy_history = []
    
    for _ in tqdm(range(num_runs), desc="Progresso da Análise"):
        all_preds, all_labels = [], []
        with torch.no_grad():
            for imgs, labels, regions in loader:
               
                transformed_imgs = torch.stack([eval_transform(img.cpu()) for img in imgs]).to(device)
                labels, regions = labels.to(device), regions.to(device)
                outputs = model(transformed_imgs, regions)
                _, preds = torch.max(outputs, 1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        report = classification_report(all_labels, all_preds, target_names=CLASS_NAMES, output_dict=True, zero_division=0)
        cm = confusion_matrix(all_labels, all_preds)
        accuracy_history.append(report['accuracy'])
        
        for i, name in enumerate(CLASS_NAMES):
            if name in report:
                results_history[name]['precision'].append(report[name]['precision'])
                results_history[name]['recall'].append(report[name]['recall'])
                results_history[name]['f1-score'].append(report[name]['f1-score'])
                tp = cm[i, i]; fp = cm[:, i].sum() - tp; fn = cm[i, :].sum() - tp; tn = cm.sum() - (tp + fp + fn)
                class_accuracy = (tp + tn) / cm.sum()
                results_history[name]['accuracy'].append(class_accuracy)

    final_metrics = {}
    for name in CLASS_NAMES:
        final_metrics[name] = {
            'precision_mean': np.mean(results_history[name]['precision']), 'precision_std': np.std(results_history[name]['precision']),
            'recall_mean': np.mean(results_history[name]['recall']), 'recall_std': np.std(results_history[name]['recall']),
            'f1-score_mean': np.mean(results_history[name]['f1-score']), 'f1-score_std': np.std(results_history[name]['f1-score']),
            'accuracy_mean': np.mean(results_history[name]['accuracy']), 'accuracy_std': np.std(results_history[name]['accuracy']),
        }
    return final_metrics, accuracy_history

final_metrics_by_class, acc_history = run_full_metric_analysis(model_to_test, test_loader_safe, NUM_RUNS)
mean_accuracy = np.mean(acc_history)
std_accuracy = np.std(acc_history)

print("\n\n" + "="*90)
print("TABELA DE PERFORMANCE DO MODELO (ANÁLISE SEGURA)")
print("="*90)
print("| Métrica | Classe | Precision | Recall | F1-Score | Acurácia (por Classe) (%) |")
print("| :--- | :--- | :--- | :--- | :--- | :--- |")
for i, name in enumerate(CLASS_NAMES):
    metrics = final_metrics_by_class[name]
    metric_group = "**Por Classe**" if i == 0 else ""
    precision_str = f"{metrics['precision_mean']:.2f} ± {metrics['precision_std']:.2f}"
    recall_str = f"{metrics['recall_mean']:.2f} ± {metrics['recall_std']:.2f}"
    f1_str = f"{metrics['f1-score_mean']:.2f} ± {metrics['f1-score_std']:.2f}"
    acc_class_str = f"{(metrics['accuracy_mean'] * 100):.2f} ± {(metrics['accuracy_std'] * 100):.2f}"
    print(f"| {metric_group} | {name} ({i}) | {precision_str} | {recall_str} | {f1_str} | {acc_class_str} |")
print("| **---** | **---** | **---** | **---** | **---** | **---** |")
acc_geral_str = f"{(mean_accuracy * 100):.2f} ± {(std_accuracy * 100):.2f}"
print(f"| **Geral** | **Acurácia** | - | - | - | **{acc_geral_str}** |")

# Célula 9 – Análise Detalhada por Região Facial

In [ ]:

print("--- Iniciando Análise de Estabilidade Completa por Grau e por Região ---")

import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
from tqdm.notebook import tqdm
from collections import defaultdict

NUM_RUNS = 30 
CLASS_NAMES = ['Grau 1', 'Grau 2', 'Grau 3', 'Grau 4'] 

model_to_test = OptimizedMultiRegionResNet18().to(device)
model_to_test.load_state_dict(torch.load("best_robust_model.pth"))
model_to_test.eval()

eval_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomRotation(15),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor()
])

def run_full_regional_stability_analysis(model, loader, num_runs):
    """
    Executa a avaliação múltiplas vezes para calcular a média e o desvio padrão
    de todas as métricas para cada CLASSE dentro de cada REGIÃO.
    """
    history = {r: {c: {'precision': [], 'recall': [], 'f1-score': [], 'accuracy': []} for c in CLASS_NAMES} for r in REGIONS}
    acc_history_regional = {r: [] for r in REGIONS}
    
    print(f"Iniciando análise com {num_runs} execuções. Isso pode levar alguns minutos...")
    
    for _ in tqdm(range(num_runs), desc="Progresso da Análise Detalhada"):
        region_results_run = defaultdict(lambda: {'y_true': [], 'y_pred': []})
        
        with torch.no_grad():
            for imgs, labels, region_indices in loader:
                transformed_imgs = torch.stack([eval_transform(img.cpu()) for img in imgs]).to(device)
                labels_dev, regions_dev = labels.to(device), region_indices.to(device)
                
                outputs = model(transformed_imgs, regions_dev)
                _, preds = torch.max(outputs, 1)
                
                for i in range(len(labels)):
                    region_name = REGIONS[region_indices[i].item()]
                    region_results_run[region_name]['y_true'].append(labels[i].item())
                    region_results_run[region_name]['y_pred'].append(preds[i].item())

        for region_name, data in region_results_run.items():
            if not data['y_true']: continue
            
            labels_range = list(range(len(CLASS_NAMES)))
            report = classification_report(data['y_true'], data['y_pred'], target_names=CLASS_NAMES, output_dict=True, zero_division=0, labels=labels_range)
            cm_region = confusion_matrix(data['y_true'], data['y_pred'], labels=labels_range)
            
            acc_history_regional[region_name].append(report['accuracy'])
            
            for i, class_name in enumerate(CLASS_NAMES):
                if class_name in report:
                    for metric in ['precision', 'recall', 'f1-score']:
                        history[region_name][class_name][metric].append(report[class_name][metric])
                    
                    tp = cm_region[i, i]
                    fp = cm_region[:, i].sum() - tp
                    fn = cm_region[i, :].sum() - tp
                    tn = cm_region.sum() - (tp + fp + fn)
                    class_accuracy = (tp + tn) / cm_region.sum() if cm_region.sum() > 0 else 0
                    history[region_name][class_name]['accuracy'].append(class_accuracy)

    final_metrics = {r: {c: {} for c in CLASS_NAMES} for r in REGIONS}
    final_acc = {}

    for region_name in REGIONS:
        final_acc[region_name] = {
            'mean': np.mean(acc_history_regional[region_name]) if acc_history_regional[region_name] else 0,
            'std': np.std(acc_history_regional[region_name]) if acc_history_regional[region_name] else 0
        }
        for class_name in CLASS_NAMES:
            for metric in ['precision', 'recall', 'f1-score', 'accuracy']: # ATUALIZADO
                values = history[region_name][class_name][metric]
                final_metrics[region_name][class_name][f'{metric}_mean'] = np.mean(values) if values else 0
                final_metrics[region_name][class_name][f'{metric}_std'] = np.std(values) if values else 0
                
    return final_metrics, final_acc

final_metrics_by_region, final_accuracy_by_region = run_full_regional_stability_analysis(model_to_test, test_loader, NUM_RUNS)

print("\n\n" + "="*90)
print("TABELAS DE ESTABILIDADE DAS MÉTRICAS POR GRAU E REGIÃO (MÉDIA ± DESVIO PADRÃO)")
print("="*90)

for region_name in REGIONS:
    print(f"\n### Tabela de Performance para a Região: {region_name.replace('_', ' ').title()}\n")
    print("| Classe | Precision | Recall | F1-Score | Acurácia (por Classe) |")
    print("| :--- | :--- | :--- | :--- | :--- |")
    
    for i, class_name in enumerate(CLASS_NAMES):
        metrics = final_metrics_by_region[region_name][class_name]
        
        precision_str = f"{metrics['precision_mean']:.2f} ± {metrics['precision_std']:.2f}"
        recall_str = f"{metrics['recall_mean']:.2f} ± {metrics['recall_std']:.2f}"
        f1_str = f"{metrics['f1-score_mean']:.2f} ± {metrics['f1-score_std']:.2f}"
        acc_class_str = f"{(metrics['accuracy_mean'] * 100):.2f}% ± {(metrics['accuracy_std'] * 100):.2f}"
        
        print(f"| {class_name} ({i}) | {precision_str} | {recall_str} | {f1_str} | {acc_class_str} |")
        
    print("| **---** | **---** | **---** | **---** | **---** |")
    
    acc_metrics = final_accuracy_by_region[region_name]
    acc_geral_str = f"{(acc_metrics['mean'] * 100):.2f}% ± {(acc_metrics['std'] * 100):.2f}"
    print(f"| **Acurácia (Geral da Região)** | - | - | - | **{acc_geral_str}** |")


--- Iniciando Análise de Estabilidade Completa por Grau e por Região ---
Iniciando análise com 30 execuções. Isso pode levar alguns minutos...


Progresso da Análise Detalhada:   0%|          | 0/30 [00:00<?, ?it/s]



TABELAS DE ESTABILIDADE DAS MÉTRICAS POR GRAU E REGIÃO (MÉDIA ± DESVIO PADRÃO)

### Tabela de Performance para a Região: Forehead

| Classe | Precision | Recall | F1-Score | Acurácia (por Classe) |
| :--- | :--- | :--- | :--- | :--- |
| Grau 1 (0) | 0.94 ± 0.02 | 0.98 ± 0.03 | 0.96 ± 0.02 | 98.46% ± 0.72 |
| Grau 2 (1) | 0.98 ± 0.02 | 0.99 ± 0.02 | 0.99 ± 0.02 | 99.35% ± 0.77 |
| Grau 3 (2) | 0.98 ± 0.06 | 0.80 ± 0.00 | 0.88 ± 0.03 | 98.82% ± 0.32 |
| Grau 4 (3) | 1.00 ± 0.00 | 1.00 ± 0.00 | 1.00 ± 0.00 | 100.00% ± 0.00 |
| **---** | **---** | **---** | **---** | **---** |
| **Acurácia (Geral da Região)** | - | - | - | **98.32% ± 0.77** |

### Tabela de Performance para a Região: Chin

| Classe | Precision | Recall | F1-Score | Acurácia (por Classe) |
| :--- | :--- | :--- | :--- | :--- |
| Grau 1 (0) | 0.92 ± 0.02 | 0.90 ± 0.01 | 0.91 ± 0.01 | 94.31% ± 0.69 |
| Grau 2 (1) | 0.92 ± 0.01 | 0.90 ± 0.02 | 0.91 ± 0.01 | 92.22% ± 0.88 |
| Grau 3 (2) | 0.88 ± 0.03 | 0.97 ± 0.02 | 0.92 ± 0.0

# Célula 10 – Conversão para Flutter (Deployment)


In [ ]:


print("--- Iniciando conversão do modelo para Pytorch Mobile (.ptl) ---")

model_to_convert = OptimizedMultiRegionResNet18()

model_to_convert.load_state_dict(torch.load("best_robust_model.pth", map_location=torch.device('cpu')))
model_to_convert.eval()

example_image = torch.rand(1, 3, 224, 224)
example_region = torch.tensor([[0]]) 

traced_script_module = torch.jit.trace(model_to_convert, (example_image, example_region))
from torch.utils.mobile_optimizer import optimize_for_mobile
optimized_traced_module = optimize_for_mobile(traced_script_module)
optimized_traced_module._save_for_lite_interpreter("acne_model.ptl")

print("\nModelo convertido para acne_model.ptl com sucesso! ✅")